# RF Diffusion Implementation
https://github.com/RosettaCommons/RFdiffusion

Newer cards may need: https://github.com/RosettaCommons/RFdiffusion/issues/349
1. conda create -n my_cuda_env python=3.11
2. conda activate my_cuda_env
3. conda install -c "nvidia/label/cuda-12.8.0" cuda-toolkit
4. pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
5. conda install -c dglteam/label/th24_cu124 dgl
6. pip install pandas
7. Install SE3Transformer as per RFdiffusion
8. Add `weights_only=False` argument to `_Jd, _W3j_flat, _W3j_indices = torch.load(os.path.join(os.path.dirname(__file__), 'constants.pt'))` in "~/miniconda3/envs/my_cuda_env/lib/python3.11/site-packages/e3nn/o3/_wigner.py"
9. pip install omegaconf hydra-core pyrsistent

## Setup

### Create Conda environment

In [2]:
!conda env create -f ../Tools/RFdiffusion/env/SE3nv.yml

2 channel Terms of Service accepted
Channels:
 - defaults
 - conda-forge
 - pytorch
 - dglteam
 - nvidia
Platform: linux-64
Solving environment: done

cudatoolkit-11.1.1   | 929.6 MB  |                                       |   0% 
dgl-cuda11.1-0.9.1po | 223.5 MB  |                                       |   0% 

mkl-2021.4.0         | 142.6 MB  |                                       |   0% 


pytorch-1.9.1        | 45.2 MB   |                                       |   0% 



scipy-1.10.1         | 23.1 MB   |                                       |   0% 




python-3.9.24        | 23.1 MB   |                                       |   0% 





torchvision-0.15.2   | 9.8 MB    |                                       |   0% 






numpy-base-1.24.3    | 6.9 MB    |                                       |   0% 







torchaudio-0.9.1     | 4.4 MB    |                                       |   0% 








intel-openmp-2021.4. | 4.2 MB    |                                       |   0% 





Please select the SE3nv Conda Environment from the Kernel Selector in VS Code

In [ ]:
# Note that these commands are listed but cannot be executed in the notebook directly.
# Use the kernel selector to activate conda. The next block of code will point to the folder directly
!conda activate SE3nv

### Use `pip` to set up packages

*`cd` command coes not work directly in VS code*

In [ ]:
import os
cur_dir = os.getcwd()
os.chdir('../Tools/RFdiffusion/env/SE3Transformer')

%pip install --no-cache-dir -r requirements.txt
!python setup.py install # Depricated

os.chdir(cur_dir)

### Install RFdiffusion

Does not want to run in VS Code. Can do setup with Conda terminal

In [3]:
os.chdir('../Tools/RFdiffusion')
%pip install -e . # install the rfdiffusion module from the root of the repository

os.chdir(cur_dir)

Obtaining file:///home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion
  Preparing metadata (setup.py) ... done
  DEPRECATION: Legacy editable install of rfdiffusion==1.1.0 from file:///home/ryangustafson/Documents/GitHub/PhD-Research/Tools/RFdiffusion (setup.py develop) is deprecated. pip 25.3 will enforce this behaviour change. A possible replacement is to add a pyproject.toml or enable --use-pep517, and use setuptools >= 64. If the resulting installation is not behaving as expected, try using --config-settings editable_mode=compat. Please consult the setuptools documentation for more information. Discussion can be found at https://github.com/pypa/pip/issues/11457
  Running setup.py develop for rfdiffusion
Note: you may need to restart the kernel to use updated packages.


## Use RFDiffusion

Once the environment is set up, just use the Kernel picker to use the environment

### Setup

In [ ]:
import os, time, subprocess, json, re, shutil

original_directory = os.getcwd()

data_path = "../Data/TIMP_Complexes/HADDOCK_PDB"
rf_diff_path = "../Tools/RFdiffusion/scripts"
protein_mpnn_path = "../Tools/ProteinMPNN/"

#pdb_complex_file_name = "TIMP3_vs_ADAM17_X_ray.pdb"
pdb_complex_file_name = "TIMP3_vs_ADAM10_HADDOCK_Xray.pdb"
pdb_path = os.path.join(data_path, pdb_complex_file_name)
output_dir = os.path.join("../Local/rfdiffusion_output", pdb_complex_file_name[:-4])
pmpnn_out_dir = "../Local/proteinmpnn_output"
output_prefix = "design"

loop_names = ["C", "EF"]
if isinstance(loop_names, str):
    loop_names = [loop_names]

loop_configs = {
    "AB": {"normal": 6, "max": 15, "pos": 30, "left": "LVK", "right": "LVY"},
    "C": {"normal": 6, "max": 15, "pos": 62, "left": "HTE", "right": "GLK"},
    "EF": {"normal": 4, "max": 10, "pos": 92, "left": "MYT", "right": "FVE"},
    "GH": {"normal": 10, "max": 20, "pos": 127, "left": "KSC", "right": "NEC"},
    "Multi": {"normal": 10, "max": 20, "pos": 143, "left": "LWT", "right": "YQS"}
}

for name in loop_names:
    if name not in loop_configs:
        raise Exception(f"Specified Loop '{name}' is unknown")

selected_loops = [loop_configs[name] for name in loop_names]
selected_loops.sort(key=lambda x: x["pos"])

chain_to_design = "A"
fixed_chains = ["B"]
total_length = 121

contig_parts = []
current_pos = 1

for loop in selected_loops:
    if current_pos <= loop["pos"]:
        contig_parts.append(f"{chain_to_design}{current_pos}-{loop['pos']}")
    contig_parts.append(f"{loop['normal']}-{loop['max']}")
    current_pos = loop["pos"] + loop["normal"] + 1

if current_pos <= total_length:
    contig_parts.append(f"{chain_to_design}{current_pos}-{total_length}")

contig_string = "/".join(contig_parts)
num_sequences_to_generate = 5

print(original_directory)
print(output_dir)
print(contig_string)

/home/ryangustafson/Documents/GitHubProj/PhD-Research/Generation
../Local/rfdiffusion_output/TIMP3_vs_ADAM10_HADDOCK_Xray
A1-62/6-15/A69-92/4-10/A97-121


In [2]:
# 3-letter to 1-letter amino acid code map
aa_map = {'CYS': 'C', 'ASP': 'D', 'SER': 'S', 'GLN': 'Q', 'LYS': 'K',
        'ILE': 'I', 'PRO': 'P', 'THR': 'T', 'PHE': 'F', 'ASN': 'N',
        'GLY': 'G', 'HIS': 'H', 'LEU': 'L', 'ARG': 'R', 'TRP': 'W',
        'ALA': 'A', 'VAL': 'V', 'GLU': 'E', 'TYR': 'Y', 'MET': 'M'}

def get_aa_sequence(pdb_file, chain_letter):
    current_sequence = []
    with open(pdb_file, 'r') as f:
        for line in f:
            line_items = line.split()
            if line.startswith('ATOM') and line_items[4] == chain_letter and line_items[2] == "CA":
                res_name = line_items[3]
                if res_name in aa_map:
                    current_sequence.append(aa_map[res_name])
    return "".join(current_sequence)

Debugging

In [22]:
os.chdir(original_directory) # if have to stop terminal run or crash before dir return
print(os.getcwd())

/home/ryangustafson/Documents/GitHub/PhD-Research/Generation


In [7]:
os.environ["HYDRA_FULL_ERROR"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

### Run RFdiffusion

In [3]:
if not pdb_path:
    print("Cannot run RFdiffusion without a scaffold PDB file.")
    raise Exception("No PDB File")

print("Preparing to run RFdiffusion...")
fix_chain_len = len(get_aa_sequence(pdb_path, fixed_chains[0]))

# Construct the command for RFdiffusion
run_command = [
    "python",
    os.path.join(rf_diff_path.replace('../', ''), "run_inference.py"),
    f"inference.output_prefix={os.path.join(output_dir.replace('../', ''), output_prefix)}",
    f"inference.input_pdb={pdb_path.replace('../', '')}",
    f'contigmap.contigs=[{contig_string}/0 {fixed_chains[0]}1-{fix_chain_len}]', #-- /0 is chain break
    "diffuser.T=20", # default 50, short sequence
    f"inference.num_designs={num_sequences_to_generate}",
]

print("Running RFdiffusion to generate novel loops and structures...")
print(" ".join(run_command))

# Run the command and stream output live
os.chdir("..") 
st = time.time()
result = subprocess.run(run_command, capture_output=True, text=True)
end = time.time()
os.chdir(original_directory)

print("--- STDOUT ---")
print(result.stdout)

print("--- STDERR ---")
print(result.stderr)

print(f"RFdiffusion finished in {(end-st)/60:.2f} minutes.")

Preparing to run RFdiffusion...
Running RFdiffusion to generate novel loops and structures...
python Tools/RFdiffusion/scripts/run_inference.py inference.output_prefix=Local/rfdiffusion_output/TIMP3_vs_ADAM10_HADDOCK_Xray/design inference.input_pdb=Data/TIMP3_vs_ADAM10_HADDOCK_Xray.pdb contigmap.contigs=[A1-62/6-15/A69-92/4-10/A97-121/0 B1-431] diffuser.T=20 inference.num_designs=5
--- STDOUT ---
[2026-03-08 11:39:31,117][__main__][INFO] - Found GPU with device_name NVIDIA GeForce RTX 5070 Ti. Will run RFdiffusion on NVIDIA GeForce RTX 5070 Ti
Reading models from /home/ryangustafson/Documents/GitHubProj/PhD-Research/Tools/RFdiffusion/rfdiffusion/inference/../../models
[2026-03-08 11:39:31,117][rfdiffusion.inference.model_runners][INFO] - Reading checkpoint from /home/ryangustafson/Documents/GitHubProj/PhD-Research/Tools/RFdiffusion/rfdiffusion/inference/../../models/Base_ckpt.pt
This is inf_conf.ckpt_path
/home/ryangustafson/Documents/GitHubProj/PhD-Research/Tools/RFdiffusion/rfdiffusi

### Analyze Results

In [4]:
# --- Parse PDB Results to Extract Sequences ---
print("parsing generated PDBs to extract sequences...")

generated_sequences = set()
generated_loops = set()
chain_id_to_extract = contig_string.split('/')[0][0] # Get chain from contig

for i in range(num_sequences_to_generate):
    pdb_file_name = f"{output_prefix}_{i}.pdb"
    pdb_file = os.path.join(output_dir, pdb_file_name)
    if os.path.exists(pdb_file):
        current_sequence = get_aa_sequence(pdb_file, chain_id_to_extract)
        generated_sequences.add(current_sequence)
        
        loop_seqs = []
        curr_idx = 0
        for loop in selected_loops:
            flank_left = loop["left"]
            flank_right = loop["right"]
            regex_pattern = re.compile(f"{flank_left}([A-Z]+?){flank_right}")
            match = regex_pattern.search(current_sequence[curr_idx:])
            if match:
                loop_seqs.append(match.group(1))
                curr_idx += match.end() - len(flank_right)
            else:
                loop_seqs.append("MISSING")
        
        if "MISSING" not in loop_seqs:
            generated_loops.add("_".join(loop_seqs))

print(f"\nExtracted {len(generated_sequences)} unique, novel loop sequences.")
print("Here are a few examples (full):")
for seq in list(generated_sequences)[:5]:
    print(f"   - {seq}")
print("Just the loops:")
for seq in list(generated_loops)[:5]:
    print(f"   - {seq}")

parsing generated PDBs to extract sequences...

Extracted 5 unique, novel loop sequences.
Here are a few examples (full):
   - CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEGGGGGGGGGGGGGGGLKLEVNKYQYLLTGRVYDGKMYTGGGGGGGGFVERWDQLTLSQRKGLNYRYHLGCN
   - CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEGGGGGGGGGGGGLKLEVNKYQYLLTGRVYDGKMYTGGGGGGGGFVERWDQLTLSQRKGLNYRYHLGCN
   - CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEGGGGGGGGLKLEVNKYQYLLTGRVYDGKMYTGGGGGGFVERWDQLTLSQRKGLNYRYHLGCN
   - CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEGGGGGGGGGLKLEVNKYQYLLTGRVYDGKMYTGGGGFVERWDQLTLSQRKGLNYRYHLGCN
   - CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEGGGGGGGGGGGGLKLEVNKYQYLLTGRVYDGKMYTGGGGGGGFVERWDQLTLSQRKGLNYRYHLGCN
Just the loops:
   - GGGGGGGGGGG_GGGGGGGG
   - GGGGGGGG_GGGG
   - GGGGGGGGGGG_GGGGGGG
   - GGGGGGG_GGGGGG
   - GGGGGGGGGGGGGG_GGGGGGGG


### ProteinMPNN

In [5]:
pmpnn_out_dir_full = os.path.join(pmpnn_out_dir, pdb_complex_file_name[:-4])
os.makedirs(pmpnn_out_dir_full, exist_ok=True)
mp_seqs = 10

print(pmpnn_out_dir_full)

for i in range(num_sequences_to_generate):
    # Create JSONL helper files for ProteinMPNN
    chain_json = {f"{output_prefix}_{i}": [[c for c in fixed_chains], [str(chain_to_design)]]}
    with open(f"{pmpnn_out_dir_full}/{output_prefix}_chain_B.jsonl", "w") as f:
        json.dump(chain_json, f)

    pdb_file_name = f"{output_prefix}_{i}.pdb"
    pdb_file = os.path.join(output_dir, pdb_file_name)
    if os.path.exists(pdb_file):
        aa_sequence = get_aa_sequence(pdb_file, chain_to_design)
    else:
        raise FileNotFoundError
    print(aa_sequence)

    fixed_positions = []
    current_fixed_start = 1
    current_seq_idx = 0
    
    for loop in selected_loops:
        flank_left = loop["left"]
        flank_right = loop["right"]
        regex_pattern = re.compile(f"{flank_left}([A-Z]+?){flank_right}")
        match = regex_pattern.search(aa_sequence[current_seq_idx:])
        if match:
            match_start = current_seq_idx + match.start()
            match_end = current_seq_idx + match.end()
            inserted_seq = match.group(1)
            loop_start_1idx = match_start + len(flank_left) + 1
            
            fixed_positions.extend(range(current_fixed_start, loop_start_1idx))
            current_fixed_start = loop_start_1idx + len(inserted_seq)
            current_seq_idx = match_end - len(flank_right)
    
    new_total_length = len(aa_sequence)
    fixed_positions.extend(range(current_fixed_start, new_total_length + 1))

    # Freeze all residues except the inpainted region
    fixed_json = {
        f"{output_prefix}_{i}": {
            str(chain_to_design): fixed_positions
        }
    }
    for c in fixed_chains:
        fixed_json[f"{output_prefix}_{i}"][c] = [i for i in range(1,fix_chain_len+1)]
    with open(f"{pmpnn_out_dir_full}/fixed.jsonl", "w") as f:
        json.dump(fixed_json, f)

    if not output_dir:
        print("Cannot run ProteinMPNN without a scaffold files.")
        raise Exception("No RFdiffusion Files")

    print("Preparing to run ProteinMPNN...")

    # Construct the command for ProteinMPNN
    run_command = [
        "python",
        os.path.join(protein_mpnn_path.replace('../', ''), "protein_mpnn_run.py"),
        "--pdb_path", pdb_file.replace('../', ''),
        "--out_folder", f"{pmpnn_out_dir_full.replace('../', '')}",
        "--chain_id_jsonl", f"{pmpnn_out_dir_full.replace('../', '')}/{output_prefix}_chain_B.jsonl",
        "--fixed_positions_jsonl", f"{pmpnn_out_dir_full.replace('../', '')}/fixed.jsonl",
        "--num_seq_per_target", f"{mp_seqs}",
        "--sampling_temp", "0.1"
    ]

    print("Running ProteinMPNN to generate sequences...")
    print(" ".join(run_command))

    # Run the command and stream output live
    os.chdir("..") 
    st = time.time()
    result = subprocess.run(run_command, capture_output=True, text=True)
    end = time.time()
    os.chdir(original_directory)

    print("--- STDOUT ---")
    print(result.stdout)

    print("--- STDERR ---")
    print(result.stderr)

    print(f"ProteinMPNN finished in {(end-st)/60:.2f} minutes.")

    os.makedirs(os.path.join(pmpnn_out_dir_full, "FA_files"), exist_ok=True)
    shutil.move(os.path.join(pmpnn_out_dir_full, "seqs", f"{output_prefix}_{i}.fa"), 
                os.path.join(pmpnn_out_dir_full, "FA_files", f"ProMPNN_from_RFd_{i}.fa"))


../Local/proteinmpnn_output/TIMP3_vs_ADAM10_HADDOCK_Xray
CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKMYRGFTKMPHVQYIHTEGGGGGGGGGGGGLKLEVNKYQYLLTGRVYDGKMYTGGGGGGGFVERWDQLTLSQRKGLNYRYHLGCN
Preparing to run ProteinMPNN...
Running ProteinMPNN to generate sequences...
python Tools/ProteinMPNN/protein_mpnn_run.py --pdb_path Local/rfdiffusion_output/TIMP3_vs_ADAM10_HADDOCK_Xray/design_0.pdb --out_folder Local/proteinmpnn_output/TIMP3_vs_ADAM10_HADDOCK_Xray --chain_id_jsonl Local/proteinmpnn_output/TIMP3_vs_ADAM10_HADDOCK_Xray/design_chain_B.jsonl --fixed_positions_jsonl Local/proteinmpnn_output/TIMP3_vs_ADAM10_HADDOCK_Xray/fixed.jsonl --num_seq_per_target 100 --sampling_temp 0.1
--- STDOUT ---
----------------------------------------
pssm_jsonl is NOT loaded
----------------------------------------
omit_AA_jsonl is NOT loaded
----------------------------------------
bias_AA_jsonl is NOT loaded
----------------------------------------
tied_positions_jsonl is NOT loaded
------------------------

### Analyze ProteinMPNN

In [8]:
import re
import csv
from pathlib import Path
import pandas as pd

# ---- USER SETTINGS ----
input_dir = Path(pmpnn_out_dir_full, "FA_files")  # directory containing your design files
output_csv = "design_summary.csv"
top_n = 10 # number of top unique loops to keep per loop length

# ---- PARSE FILES ----
records = []

print(input_dir)

for file in input_dir.glob("*.fa"):
    with open(file) as f:
        lines = [line.strip() for line in f if line.strip()]

    for i in range(0, len(lines), 2):
        header = lines[i]
        seq = lines[i+1]
        if "/" in seq:
            seq = seq.split("/")[0]

        # extract metadata
        entry = {"file": file.name, "full_seq": seq}
        parts = header.strip(">").split(", ")
        for p in parts:
            if "=" in p:
                key, val = p.split("=", 1)
                entry[key.strip()] = val.strip()
            else:
                entry["design_type"] = p.strip()

            # extract loop region between flanks
            loop_seqs = []
            curr_idx = 0
            for loop in selected_loops:
                f_left = loop["left"]
                f_right = loop["right"]
                m = re.search(f"{f_left}(.*?){f_right}", seq[curr_idx:])
                if m:
                    loop_seqs.append(m.group(1))
                    curr_idx += m.end() - len(f_right)
                else:
                    loop_seqs.append("MISSING")
            
            if "MISSING" not in loop_seqs:
                entry["loop_seq"] = "_".join(loop_seqs)
                entry["loop_length"] = sum(len(ls) for ls in loop_seqs)
            else:
                entry["loop_seq"] = None
                entry["loop_length"] = None

        records.append(entry)

# ---- MAKE DATAFRAME ----
df = pd.DataFrame(records)

print(df)

# Convert numeric columns
for col in ["score", "global_score", "seq_recovery", "T"]:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

# ---- REORDER COLUMNS ----
column_order = [
    "file", "sample", "loop_seq", "loop_length",
    "score", "global_score", "seq_recovery",
    "design_type", "full_seq", "fixed_chains", "designed_chains",
    "model_name", "git_hash", "seed", "T"
]
df = df[[c for c in column_order if c in df.columns]]

# ---- SAVE FULL SUMMARY ----
df.to_csv(os.path.join(pmpnn_out_dir_full, output_csv), index=False)
print(f"Saved {len(df)} entries to {output_csv}")

# ---- SUMMARIZE BEST UNIQUE LOOPS ----
clean_df = df.dropna(subset=["loop_seq", "score"]).copy()

# Keep best (lowest score) per unique loop sequence
best_per_loop = (
    clean_df.sort_values("score")
    .groupby("loop_seq", as_index=False)
    .first()
)

# Compute averages per loop length
avg_stats = (
    clean_df.groupby("loop_length", as_index=False)
    .agg(
        avg_score=("score", "mean"),
        avg_seq_recovery=("seq_recovery", "mean"),
        count=("loop_seq", "count")
    )
    .sort_values("loop_length")
)

# Select top N best loops per loop length
best_per_length = (
    best_per_loop.sort_values(["loop_length", "score"])
    .groupby("loop_length", group_keys=False)
    .head(top_n)
)

# Merge in average stats
best_per_length = best_per_length.merge(avg_stats, on="loop_length", how="left")

# Sort final summary
best_per_length = best_per_length.sort_values(["loop_length", "score"], ascending=[True, True])

# ---- SAVE SUMMARIES ----
best_per_length.to_csv(os.path.join(pmpnn_out_dir_full, "best_loops_per_length.csv"), index=False)
avg_stats.to_csv(os.path.join(pmpnn_out_dir_full, "loop_length_averages.csv"), index=False)

print(f"Saved best {top_n} unique loops per length to best_loops_per_length.csv")
print("Saved loop length averages to loop_length_averages.csv")


../Local/proteinmpnn_output/TIMP3_vs_ADAM10_HADDOCK_Xray/FA_files
                      file                                           full_seq  \
0    ProMPNN_from_RFd_1.fa  CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKM...   
1    ProMPNN_from_RFd_1.fa  CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKM...   
2    ProMPNN_from_RFd_1.fa  CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKM...   
3    ProMPNN_from_RFd_1.fa  CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKM...   
4    ProMPNN_from_RFd_1.fa  CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKM...   
..                     ...                                                ...   
500  ProMPNN_from_RFd_3.fa  CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKM...   
501  ProMPNN_from_RFd_3.fa  CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKM...   
502  ProMPNN_from_RFd_3.fa  CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKM...   
503  ProMPNN_from_RFd_3.fa  CTCSPSHPQDAFCNSDIVIRAKVVGKKLVKEGPFGTLVYTIKQMKM...   
504  ProMPNN_from_RFd_3.fa  CTCSPSHPQDAFCNS

In [ ]:
# Final cleanup
original_sequences = set(df['sequence'])
unique_new_sequences = list(generated_sequences - original_sequences)

print(f"\nExtracted {len(unique_new_sequences)} unique, novel loop sequences.")
print("Here are a few examples:")
for seq in unique_new_sequences[:5]:
    print(f"   - {seq}")